In [ ]:
!pip install geemap

In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-climatesentinel')

Case study: Assam Floods 2024

In [ ]:
# Initialise
ee.Initialize(project='ee-climatesentinel')

# Define the region - Dibrugarh/Majuli Area, Assam
# format will be -  [min_lon, min_lat, max_lon, max_lat]
roi = ee.Geometry.Rectangle([93.5, 26.5, 95.5, 27.5])

# Write function to FETCH and CLEAN SAR data
def get_sar_image(start_date, end_date):
  collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(roi)
  .filterDate(start_date, end_date)
  # Flood mapping we use VH polarisation
  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
  .filter(ee.Filter.eq('instrumentMode', 'IW'))
  .select ('VH'))

  # Use mean to reduce noise
  return collection.mean().clip(roi)

# Now fetch BEFORE and AFTER flood images
# Pre-flood: May 2024
pre_flood = get_sar_image('2024-05-01', '2024-05-25')
# Post-flood: June 2024 (Peak flood)
post_flood = get_sar_image('2024-06-15', '2024-06-30')

# The Change Detection (The "Sentinel" Logic)
# If the ratio of Post/Pre is low, it means land turned into water
flood_ratio = post_flood.divide(pre_flood)
# Threshold: 0.8 is the standard "water" drop-off point
flood_mask = flood_ratio.lt(0.8).selfMask()

# 6. Create the Interactive Map
Map = geemap.Map()
Map.centerObject(roi, 9)

# Add the SAR layers (Visualized in dB scale: -25 to 0)
Map.addLayer(pre_flood, {'min': -25, 'max': 0}, '1. Pre-Flood (May)')
Map.addLayer(post_flood, {'min': -25, 'max': 0}, '2. Post-Flood (June)')

# Add the Flood Mask in bright Red
Map.addLayer(flood_mask, {'palette': ['red']}, '3. DETECTED FLOOD')

Map

Calculated detected flood areas (in Hectare) and total flood area (in sq. km.)

In [ ]:
# Get area of each individual pixel in square meters
area_image = flood_mask.multiply(ee.Image.pixelArea())

# Sum up all the "Flood" pixels within the ROI
stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=10, # Sentinel-1 resolution is 10m
    maxPixels=1e10
)

# Convert from Square Meters to Hectares (1 Hectare = 10,000 m2)
area_m2 = stats.getNumber('VH').getInfo()
area_ha = area_m2 / 10000

print(f"SENTINEL REPORT")
print(f"Total Flooded Area Detected: {area_ha:,.2f} Hectares")
print(f"Total Flooded Area (Sq Km): {area_ha/100:,.2f} km²")